
# Supervised Learning Algorithms - Classification

In this notebook, we'll explore the most common algorithms for classification. 

>
> ℹ️ Info about this notebook:
>
> - Goal: explore the most common classification algorithms, how they work, their advantages and limitations.
> - Dataset: Titanic Dataset
> - Metric: F1-Score
> 




---
## Initial Setup

We'll use the **Titanic dataset** — a classic classification problem.  
The goal: **predict whether a passenger survived** (1) or not (0).

### Features we'll use
| Feature | Description |
|---|---|
| `pclass` | Passenger class (1, 2, 3) |
| `sex` | Gender (encoded as 0/1) |
| `age` | Age in years |
| `fare` | Ticket fare |

We'll keep preprocessing minimal — the focus here is on the different algorithms, not feature engineering.

In [1]:
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Load dataset
df = sns.load_dataset("titanic")

# Minimal cleaning
df = df[["survived", "pclass", "sex", "age", "fare"]].dropna()
df["sex"] = df["sex"].map({"male": 0, "female": 1})

df.head()

,survived,pclass,sex,age,fare
0,0,3,0,22.0,7.2500
1,1,1,1,38.0,71.2833
2,1,3,1,26.0,7.9250
3,1,1,1,35.0,53.1000
4,0,3,0,35.0,8.0500


In [2]:
# Define features and target
X = df[["pclass", "sex", "age", "fare"]]
y = df["survived"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

display(f"X_train shape... {X_train.shape}")
display(f"X_test shape...  {X_test.shape}")
display(f"y_train shape... {y_train.shape}")
display(f"y_test shape...  {y_test.shape}")

'X_train shape... (571, 4)'

'X_test shape...  (143, 4)'

'y_train shape... (571,)'

'y_test shape...  (143,)'

---
## The metric we'll use for this demo: F1-score

For this notebook, we'll use the **F1-score** to compare all models in a consistent, simple way.

> **Reminder:** F1-score is the harmonic mean of Precision and Recall:
> $$F1 = 2 \cdot \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$
> - F1-score is mainly used when you want a single metric that balances precision and recall equally.
> - Score ranges from **0.0 (worst)** to **1.0 (perfect)**.

The focus here is on **exploring different algorithms** — not on maximizing performance. We'll use default hyperparameters unless otherwise noted.

In [3]:
from sklearn.metrics import f1_score

results = {}  # will store F1-scores for all models

def evaluate(model, model_name, X=None):
    X_eval = X if X is not None else X_test
    y_pred = model.predict(X_eval)
    score = f1_score(y_test, y_pred)
    results[model_name] = score
    print(f"F1-score ({model_name}): {score:.4f}")


---
## 1. Linear Models

**Core idea:** Find a straight-line (or flat hyperplane) decision boundary that separates the classes.

Linear models learn a weighted combination of features:
$$\hat{y} = w_1 x_1 + w_2 x_2 + \ldots + w_n x_n + b$$

They are simple, fast, and interpretable — but they can only capture **linear relationships** between features and the target (unless extended, e.g. with polynomial features).

### 1.1 Logistic Regression

**Intuition:** Despite the name, Logistic Regression is a *classification* algorithm. It fits a **sigmoid curve** to the data and predicts the probability of belonging to a class.

**How it works:**
1. Computes a linear combination of features: $z = w \cdot x + b$
2. Passes it through the sigmoid function: $P(y=1) = \frac{1}{1 + e^{-z}}$
3. Predicts class 1 if $P(y=1) \geq 0.5$, else class 0

The decision boundary is a **straight line** (or hyperplane in higher dimensions).

| ✅ Strengths | ❌ Weaknesses |
|---|---|
| Fast and simple | Can't capture non-linear patterns |
| Outputs probabilities | Assumes linear decision boundary |
| Great baseline model | Sensitive to outliers |
| Highly interpretable | May underfit if the data includes complex, non-linear patterns |

In [4]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

evaluate(model, "Logistic Regression")

F1-score (Logistic Regression): 0.6957


### 1.2 Logistic Regression + Polynomial Features

**Intuition:** Standard Logistic Regression can only draw straight-line boundaries. By adding **polynomial features** (e.g. $x_1^2$, $x_2^2$, $x_1 \cdot x_2$), we allow the model to fit **curved** decision boundaries — while still using a linear model under the hood.

**How it works:**
1. `PolynomialFeatures` creates new features from combinations of existing ones (degree 2 → adds squared and interaction terms)
2. Logistic Regression is then trained on this expanded feature set

| ✅ Strengths | ❌ Weaknesses |
|---|---|
| Captures non-linear patterns | Feature space grows fast with degree |
| Simple extension of Logistic Regression | Risk of overfitting with high degree |
| Still interpretable | Slower with many features |

In [5]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

scaler = StandardScaler()
X_train_poly = scaler.fit_transform(X_train_poly)
X_test_poly = scaler.transform(X_test_poly)

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_poly, y_train)

evaluate(model, "Polynomial Logistic Regression", X_test_poly)


F1-score (Polynomial Logistic Regression): 0.6792


### 1.3 Logistic Regression + Regularization (Ridge / Lasso)

**Intuition:** When a model overfits, it assigns very large weights to some features. **Regularization** adds a penalty for large weights, forcing the model to stay simpler.

**How it works:**
- **Ridge (L2):** Penalizes the sum of *squared* weights → shrinks all weights, but rarely to zero
- **Lasso (L1):** Penalizes the sum of *absolute* weights → can shrink weights **all the way to zero** (acts as feature selection)

In scikit-learn's `LogisticRegression`, regularization strength is controlled by `C`:
- **`C` = 1/λ** → smaller `C` means **stronger regularization**

| ✅ Strengths | ❌ Weaknesses |
|---|---|
| Reduces overfitting | Adds a hyperparameter to tune (`C`) |
| L1 performs automatic feature selection | L1 can be unstable with correlated features |
| Works well on high-dimensional data | May underfit if `C` is too small |

In [6]:
# Ridge (L2 regularization)
model_ridge = LogisticRegression(penalty="l2", C=0.1, random_state=42, max_iter=1000)
model_ridge.fit(X_train, y_train)
evaluate(model_ridge, "Logistic Regression + Ridge (L2)")

# Lasso (L1 regularization) — requires a solver that supports L1
model_lasso = LogisticRegression(penalty="l1", C=0.1, solver="liblinear", random_state=42)
model_lasso.fit(X_train, y_train)
evaluate(model_lasso, "Logistic Regression + Lasso (L1)")

F1-score (Logistic Regression + Ridge (L2)): 0.6789
F1-score (Logistic Regression + Lasso (L1)): 0.6786


---
## 2. Distance-Based Models

**Core idea:** Classify a data point based on **how similar it is to other known points**. Instead of learning a fixed model, these algorithms use the training data directly at prediction time.

The key concept is a **distance metric** (usually Euclidean distance) to measure how close two points are in feature space.

### 2.1 K-Nearest Neighbors (KNN)

**Intuition:** "You are who you hang out with." To classify a new point, look at its **K closest neighbors** in the training set and take a **majority vote**.

**How it works:**
1. For each test point, compute the distance to all training points
2. Select the K nearest neighbors
3. Predict the most common class among those K neighbors

**Choosing K:**
| K value | Effect |
|---|---|
| Too small (e.g. K=1) | Complex, jagged boundary → **overfitting** |
| Too large (e.g. K=100) | Very smooth boundary → **underfitting** |
| Just right | Balanced generalization |

<br>

> ⚠️ KNN requires **feature scaling** — features on different scales will distort distances.

<br>

| ✅ Strengths | ❌ Weaknesses |
|---|---|
| Simple and intuitive | Slow at prediction (compares to all training points) |
| No training phase | Poor performance in high dimensions |
| Naturally handles non-linear boundaries | Sensitive to feature scale and irrelevant features |
| Works well with small datasets | Needs more memory (stores all training data) |

In [7]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# KNN requires feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train_scaled, y_train)

evaluate(model, "KNN (k=5)", X_test_scaled)


F1-score (KNN (k=5)): 0.7193


---
## 3. Tree-Based Models

**Core idea:** Classify data by learning a sequence of **yes/no questions** about the features. Each question splits the data into two groups, forming a tree structure.

Tree-based models are intuitive, require no feature scaling, and can naturally capture **non-linear patterns** and **feature interactions**.

### 3.1 Decision Trees

**Intuition:** Split the data step-by-step by asking the question that best separates the classes at each step (e.g. *"Is the passenger female?"*, *"Is the fare > 50?"*).

**How it works:**
1. At each node, find the feature + threshold that best splits the data (using **Gini impurity** or **entropy**)
2. Repeat recursively on each resulting subset
3. Stop when a stopping criterion is met (e.g. max depth, min samples per leaf)

The result is a tree of decisions that leads to a class prediction at each leaf.

| ✅ Strengths | ❌ Weaknesses |
|---|---|
| Highly interpretable — you can visualize it | Prone to **overfitting** (especially deep trees) |
| No need for feature scaling | **High variance** — small data changes → very different tree |
| Handles non-linear patterns naturally | Strong baseline, but ensembles often perform better (most balanced) |
| Handles mixed feature types | |

In [8]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

evaluate(model, "Decision Tree")

F1-score (Decision Tree): 0.6306


---
## 4. Ensemble Methods

**Core idea:** Instead of relying on a single model, combine **many models** to produce a better prediction.

> "The wisdom of the crowd" — many weak models together can form a strong one.

There are three main strategies:

| Strategy | Idea |
|---|---|
| **Bagging** | Train models in *parallel* on random subsets of data; combine by voting |
| **Boosting** | Train models *sequentially*, each correcting the errors of the previous |
| **Stacking** | Train different models; use their predictions as input to a meta-model |

### 4.1 Simple Ensembles — Majority Vote

The simplest form of ensembling: train multiple different models and combine their predictions. 

For classification, a common approach is majority voting: each model votes for a class → the class with the most votes wins (majority vote).

> In scikit-learn: `VotingClassifier`

This approach is straightforward but rarely outperforms more sophisticated ensembles like Random Forest or Boosting.

### 4.2 Bagging → Random Forest

**Intuition:** Build many decision trees, each trained on a **random sample** of the data and a **random subset of features**. Combine their predictions via majority vote.

**How it works:**
1. Draw `n` bootstrap samples (random samples with replacement) from training data
2. For each sample, train a decision tree — but at each split, only consider a random subset of features
3. At prediction time, each tree votes → take the majority

This randomness **decorrelates** the trees, so their errors cancel each other out.

| ✅ Strengths | ❌ Weaknesses |
|---|---|
| Reduces overfitting vs a single tree | Less interpretable than a single tree |
| Robust to noise and outliers | Slower to train and predict |
| Provides feature importance scores | Higher memory usage |
| Works well out of the box | |

In [9]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

evaluate(model, "Random Forest")

F1-score (Random Forest): 0.6964


### 4.3 Boosting

**Intuition:** Train models **sequentially**. Each new model focuses on the mistakes of the previous ones — gradually improving where the ensemble is weak.

**How it works:**
1. Train a weak model (e.g. a shallow tree)
2. Compute the errors (residuals)
3. Train the next model to predict those errors
4. Repeat, adding each new model to the ensemble (with a learning rate to control contribution)
5. Final prediction = weighted sum of all models

**Popular boosting libraries:**
- **XGBoost** — fast, regularized, widely used
- **LightGBM** — very fast, great for large datasets
- **CatBoost** — handles categorical features natively

| ✅ Strengths | ❌ Weaknesses |
|---|---|
| Often state-of-the-art on tabular data | Many hyperparameters to tune |
| Handles missing values (XGBoost) | More prone to overfitting if not tuned |
| Built-in regularization | Slower to train than Random Forest |
| Strong feature importance | Harder to interpret |

In [10]:
#
# IMPORTANT: you may need to install xgboost
#
# pip install xgboost   ← uncomment and run if not installed
#
# %pip install xgboost -q
#

from xgboost import XGBClassifier

model = XGBClassifier(random_state=42, eval_metric="logloss")
model.fit(X_train, y_train)

evaluate(model, "XGBoost")

F1-score (XGBoost): 0.6972


### 4.4 Stacking

**Stacking** (or *stacked generalization*) takes ensembling one step further:

1. Train several different **base models** (e.g. Logistic Regression, KNN, Decision Tree)
2. Use their predictions as **input features** for a second-level **meta-model**
3. The meta-model learns which base models to trust and how to combine them

> In scikit-learn: `StackingClassifier`

Stacking can squeeze out extra performance, but it's more complex to set up and tune, and the gains are often marginal over a well-tuned boosting model. It's less commonly used in practice.

---
## 5. Other Algorithms

We've covered the most widely-used algorithms, but the ML landscape is much broader. Here are a few other notable ones:

---

### Support Vector Machines (SVM)

**Idea:** Find the **hyperplane that maximizes the margin** between the two classes. The data points closest to the boundary are the *support vectors*.

Using the **kernel trick**, SVMs can handle non-linear boundaries by mapping data into a higher-dimensional space.

**When to use:** Works well on small-to-medium datasets with clear margins. Can be slow on large datasets.

---

### Naive Bayes

**Idea:** A probabilistic model based on **Bayes' theorem**. It assumes all features are **independent** of each other (the "naive" assumption).

$$P(y | x_1, \ldots, x_n) \propto P(y) \prod_{i} P(x_i | y)$$

**When to use:** Very fast, works well for text classification (spam detection, sentiment analysis). The independence assumption is often wrong but the model still performs surprisingly well.

---

### Neural Networks & Deep Learning

**Idea:** Layers of interconnected nodes (neurons) that learn increasingly complex representations of the data. Deep Learning = Neural Networks with many layers.

**When to use:**
- Primarily for **unstructured data**: images, text, audio, video
- Typically needs **large amounts of data** to outperform classical ML
- For tabular/structured data, tree-based models (Random Forest, XGBoost) often perform just as well with far less complexity

> 💡 Neural Networks and Deep Learning will be covered in depth in a dedicated module.

---
## Summary: Model Comparison

Let's compare the F1-scores of all models we trained.

In [11]:
import pandas as pd

summary = (
    pd.DataFrame(list(results.items()), columns=["Model", "F1-Score"])
    .sort_values("F1-Score", ascending=False)
    .reset_index(drop=True)
)
summary.index += 1
summary

,Model,F1-Score
1,KNN (k=5),0.719298
2,XGBoost,0.697248
3,Random Forest,0.696429
4,Logistic Regression,0.695652
5,Polynomial Logistic Regression,0.679245
6,Logistic Regression + Ridge (L2),0.678899
7,Logistic Regression + Lasso (L1),0.678571
8,Decision Tree,0.630631
